In [1]:
import pdfplumber
import os
import re
import pandas as pd

In [8]:
def parse_invoice(file):
    INVOICE_NUMBER_PATT = re.compile(r'(CD2026[A-Z0-9-]+)', re.IGNORECASE)
    TOTAL_AMOUNT_PATT = re.compile(r'(?:Invoice\s*amount)\s*([\d\s.,-]*)', re.IGNORECASE)
    HS_CODE_PATT = re.compile(r'\b([\d]{8,10})\b', re.IGNORECASE)
    WEIGHT_PATT = re.compile(r'([\d.,]+)\s(?:kg)')
    page_text = ''

    data = {
        "Invoice number": None,
        "Total amount": None,
        "Items": []
    }
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            page_text += page.extract_text(keep_blank_chars = True) or '-- TEXT NOT FOUND --'
    
    invoice_found = INVOICE_NUMBER_PATT.search(page_text)
    total_found = TOTAL_AMOUNT_PATT.search(page_text)

    if invoice_found: data['Invoice number'] = invoice_found.group(1).strip()
    if total_found:
        total_string = total_found.group(1).strip()
        clean_string = re.sub(r'[^\d.-]','', total_string)
        data['Total amount'] = float(clean_string) if clean_string else 0.00

    for line in page_text.split('\n'):
        if re.match(r'\b[\d]{8,10}\b', line):
            hs = HS_CODE_PATT.search(line)
            weight = WEIGHT_PATT.search(line)
            weight_str = weight.group(1).strip() if weight else 0
            clean_weight = re.sub(r'[^\d.]', '', weight_str) 
            data['Items'].append({'HS': hs.group(1).strip() if hs else '-- HS not found --', 'Weight': float(clean_weight)})

    return data


In [9]:
def parse_customs_invoice(file):
    ORDER_PATT = re.compile(r'([A-Z]\d{6})', re.IGNORECASE)
    TOTAL_PATT = re.compile(r'(?:TotalExclusiveTax)\s*([\d.,-]+)', re.IGNORECASE)
    HS_PATT = re.compile(r'\b([\d]{8,10})\b-(?:[A-Z,a-z ]+),', re.IGNORECASE)
    WEIGHT_PATT = re.compile(r',(\d+(?:[.,]\d+)*)\s*(?:KG|kg)', re.IGNORECASE)

    full_text = ''

    data = {
        "Invoice number": None,
        "Total amount": None,
        "Items": []
    }

    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            full_text += page.extract_text(keep_blank_chars=True) or '-- Customs Invoice text not found --'

    order_found = ORDER_PATT.search(full_text)
    total_amount_found = TOTAL_PATT.search(full_text)

    if order_found: data['Invoice number'] = order_found.group(1).split()

    if total_amount_found:
        total_amount_string = total_amount_found.group(1).strip() if total_amount_found else '0'
        total_amount_clear = re.sub(r'[^[\d.-]]', '', total_amount_string)
        data['Total amount'] = float(total_amount_clear)

    for line in full_text.split('\n'):
        if re.match(HS_PATT, line):
            hs_code_found = HS_PATT.search(line)
            hs_code = hs_code_found.group(1).strip() if hs_code_found else '-- HS code not found --'
            
            weight_found = WEIGHT_PATT.search(line)
            weight_string = weight_found.group(1).strip() if weight_found else '0'
            weight_clean = re.sub(r'[^\d.]+', '', weight_string)

            data['Items'].append({
                'HS': hs_code, 'Weight': float(weight_clean)})
        
    return data

In [4]:
path = 'files'

In [5]:
result = []

In [11]:
for file_name in os.listdir(path):
    if file_name.endswith('.pdf'):
        file_path = os.path.join(path, file_name)
        try:
            if file_name.startswith('CD2026'):
                fetched_data = parse_invoice(file_path)
        
            if file_name.startswith('Customs invoice'):
                fetched_data = parse_customs_invoice(file_path)

            for i, item in enumerate(fetched_data['Items']):
                result.append({
                    "Invoice number": fetched_data['Invoice number'],
                    "Total amount": fetched_data['Total amount'] if i == 0 else None,
                    "HS Code": item['HS'],
                    "Weight": item['Weight']
                })
        except Exception as e:
            print(f'Error reading file {file_name}: {e}')

In [18]:
def write_to_excel(data, output_file):
    """
    Записывает данные в Excel-файл.
    
    Args:
        data (list): Список словарей с данными для записи.
        output_file (str): Имя выходного Excel-файла.
    """
    df = pd.DataFrame(data)

    columns_order = ['Invoice number', 'HS Code', 'Weight', 'Total amount']
    df = df[columns_order]

    df.loc['Total'] = df[['Weight', 'Total amount']].sum(numeric_only=True)

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Sheet1', index=True)
        worksheet = writer.sheets['Sheet1']

        number_format = '#,##0.00'

        total_amount_column = 4
        weight_column = 5

        for row in range(2, len(df) + 2):
            worksheet.cell(row=row, column=total_amount_column).number_format = number_format
            worksheet.cell(row=row, column=weight_column).number_format = number_format


In [19]:
output_file = 'parsed_data_notebook.xlsx'
write_to_excel(result, output_file)